# 04 — Protocolo de evaluación: walk-forward causal, caché y ejecución del benchmark

Este notebook fija **cómo** se juzga a todos los detectores, de la misma manera y sin
mirar el futuro: el protocolo *walk-forward* expandente con propagación de estado entre
refits (ADR-003), el gate previo de datos y dependencias, la caché verificada por huella
de contenido y la ejecución reanudable, secuencial o en paralelo, de la matriz
pista × detector. Es el contrato común que consumen los notebooks de familia
(`05_familia_F1_reglas` … `11_familia_F7_deep`) y la comparativa global
(`12_comparativa.ipynb`). **Aquí no se interpretan resultados.**

**Índice**

1. Protocolo walk-forward causal — 1.1 Propagación de estado entre refits (ADR-003)
2. Qué se va a probar (configuración declarada de los doce detectores)
3. Cómo se implementa la comparación
4. Caché con huella de contenido
5. Interruptores `EJECUTAR` / `CACHE_ONLY`
6. Gate previo: datos, columnas y dependencias
7. Ejecución reanudable y CLI paralela
8. Inventario de resultados disponibles
9. Cierre

---

Este notebook abre la **Fase D**: toma los paneles causales producidos por
`03_preprocesado.ipynb` y reevalúa los doce detectores de la Capa 1 bajo un
protocolo común. La lógica de la Capa 1 no se modifica; sus implementaciones viven
en `regimenes.detectores` (una subcarpeta por familia), se configuran en
`regimenes.detectores.registry`, el protocolo está en `regimenes.evaluacion` y la
ejecución en `regimenes.benchmark`.

> **Contexto.** Este notebook pertenece a la Fase 1 del TFM (régimen de mercado, ADR-005): el
> objetivo de la fase es una señal de régimen para los agentes, no un sistema de detección como
> producto final.
>
> **Objetivo de este notebook.** Ejecutar y guardar evidencia comparable. Aquí no
> se elige todavía un ganador ni se diseña el híbrido final; esa lectura se hace en
> los notebooks de familia (05–11) y en `12_comparativa.ipynb`.

**Entradas:** `data/processed/pista{A,B}_{diaria,mensual}.parquet`,
`configs/benchmark_spec.yaml` y los detectores de `regimenes.detectores`.

**Salidas:** una fila de métricas y un panel OOS por combinación pista×detector,
además de `manifest.json` y `run_status.csv` en `results/benchmark/`.

In [1]:
import warnings
from math import ceil

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

from regimenes import benchmark as bm
from regimenes import evaluacion as ev
from regimenes.benchmark import cache as bm_cache
from regimenes.detectores import detector_specs, specs_table
from regimenes.rutas import ROOT, RESULTS_BENCHMARK

pd.set_option('display.max_colwidth', 90)
print('ROOT:', ROOT)
print('Paneles procesados disponibles:', bm.processed_available())

ROOT: C:\Users\Usuario\Downloads\PRUEBAS_DETECCION_REGIMENES_DE_MERCADO_TFM
Paneles procesados disponibles: True


## 1. Protocolo walk-forward causal

Todos los detectores se evalúan con la misma función, `regimenes.evaluacion.walk_forward`,
que simula el uso en tiempo real:

1. **Primer corte.** Se reservan `train_inicial` sesiones (8 años en A, 3 en B;
   `DEFAULT_TRAIN_DAYS`) que nunca se puntúan.
2. **Refit periódico.** En cada fold se crea una instancia **nueva** del detector
   (`spec.factory()`), se ajusta solo con datos anteriores al bloque y se predicen las
   `step` sesiones siguientes (21 en general; 63 en D05; 126 en D08/D11). La ventana es
   expandente salvo D11, que usa ventana móvil del tamaño del train inicial.
3. **Orden económico causal.** Tras cada ajuste se re-fija qué estado es "crisis"
   con los retornos del S&P 500 **del train** (`label_states_economically`), de modo
   que la etiqueta `n_states − 1` significa lo mismo en todos los folds y detectores.
4. **Predicción online.** El bloque se etiqueta con `predict_online` (filtrado causal,
   nunca Viterbi/suavizado), arrancando desde la cola del train (§1.1).
5. **Una sola predicción por fecha.** Cada fecha OOS aparece una vez: la del fold que
   la cubrió por primera vez. Esas etiquetas son las únicas que consumen las métricas.
6. **Diagnóstico aislado.** La re-predicción del bloque anterior con el modelo
   siguiente (que ha visto más datos) se guarda aparte, en `attrs['stability_panel']`,
   y solo la usa `label_stability`; nunca entra en cobertura, falsas alarmas ni
   detección.

La tabla siguiente materializa ese calendario para cada pista y detector a partir de
`regimenes.detectores.registry`. Si los paneles de `data/processed/` están disponibles
se añaden el rango OOS común y el número de refits; si no, solo la configuración.

In [2]:
filas = []
for track in ('A', 'B'):
    idx = bm.common_track_index(track) if bm.processed_available(track) else None
    for spec in detector_specs(track):
        train = bm.DEFAULT_TRAIN_DAYS[track]
        fila = {
            'pista': track, 'id': spec.detector_id, 'train_inicial': train,
            'step': spec.step,
            'ventana': 'expandente' if spec.expanding else f'móvil ({train})',
        }
        if idx is not None:
            n_oos = len(idx) - train
            fila.update({
                'contexto_filas': ev.resolve_context('auto', idx),
                'oos_inicio': idx[train].date(), 'oos_fin': idx[-1].date(),
                'n_oos': n_oos, 'n_refits': ceil(n_oos / spec.step),
            })
        filas.append(fila)
calendario = pd.DataFrame(filas)
display(calendario)
if 'n_refits' in calendario:
    for track, part in calendario.groupby('pista'):
        print(f"Pista {track}: OOS {part.oos_inicio.iloc[0]} → {part.oos_fin.iloc[0]} "
              f"({part.n_oos.iloc[0]} sesiones); refits entre {part.n_refits.min()} y "
              f"{part.n_refits.max()}; contexto {part.contexto_filas.iloc[0]} filas.")
else:
    print('Sin paneles procesados: solo configuración (ver gate de §6).')

,pista,id,train_inicial,step,ventana,contexto_filas,oos_inicio,oos_fin,n_oos,n_refits
0,A,D01,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
1,A,D02,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
2,A,D03,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
3,A,D04,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
4,A,D05,2016,63,expandente,252,1970-05-12,2026-05-29,14133,225
5,A,D06,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
6,A,D07,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
7,A,D08,2016,126,expandente,252,1970-05-12,2026-05-29,14133,113
8,A,D09,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673
9,A,D10,2016,21,expandente,252,1970-05-12,2026-05-29,14133,673


Pista A: OOS 1970-05-12 → 2026-05-29 (14133 sesiones); refits entre 113 y 673; contexto 252 filas.
Pista B: OOS 2010-04-12 → 2026-05-29 (4059 sesiones); refits entre 33 y 194; contexto 252 filas.


### 1.1 Propagación de estado entre refits (ADR-003)

Antes de ADR-003 el protocolo no pasaba contexto al predecir un bloque OOS: el
detector recién ajustado empezaba el bloque en su estado inicial ("calma") salvo que él
mismo antepusiera el train como *burn-in*. Para los detectores sin memoria (GMM, AE+GMM)
es inocuo. Entre los que tienen estado hay que separar dos casos (ADR-003 §2.2):

- **Perdían la crisis en curso en cada borde de bloque** y la volvían a declarar días
  después: los **autómatas de histéresis** de D01, D02, D06 y D10, que reiniciaban en
  calma, y el Jump Model (D09), que arrancaba cada bloque sin historia.
- **No la perdían**: los filtros *forward* de los HMM (D04, D08), del Markov-switching
  (D05) y del MS-GARCH (D11) y el CUSUM de D07 ya se re-ejecutaban sobre el train
  anterior al bloque (*burn-in*), así que su estado al inicio del bloque reflejaba la
  historia real. ADR-003 lo comprueba: D07 da el mismo panel byte a byte y los HMM, MS y
  D11 no cambian. (D06 y D10 anteponían *burn-in* a su señal —σ GARCH, distancia de
  Mahalanobis—, pero no a su autómata de decisión, que sí se reiniciaba.)

Desde ADR-003, `walk_forward(..., context='auto')` predice cada bloque sobre
**[cola del train] + bloque** y conserva solo las predicciones del bloque:

- `context='auto'` = un año en la frecuencia del índice (252 filas diarias, 12
  mensuales); `None` = todo el train; `0` = protocolo anterior (arranque en frío).
- Es **causal**: las filas de contexto son anteriores al bloque y ya formaban parte del
  train del fold (tests de truncado en `tests/test_walkforward_context.py`).
- Los detectores que ya anteponían el train como *burn-in* (D04–D08, D10, D11) no
  duplican nada: recortan su burn-in a las fechas anteriores a la entrada.

La celda siguiente lo ilustra sin tocar `results/`: aplica la regla de histéresis de D01
(`RuleVixThreshold`) a una serie **sintética** de VIX con episodios de estrés largos y
compara el protocolo en frío (`context=0`) con el vigente (`context='auto'`). La
columna clave es `salidas_en_borde_de_bloque`: salidas de crisis que ocurren
exactamente el primer día de un bloque, es decir, provocadas por el reinicio y no por
los datos.

In [3]:
from regimenes.detectores.f1_reglas.rule_vix_threshold import RuleVixThreshold

# Serie sintética: VIX z-scoreado con episodios de estrés largos (AR(1) + saltos de nivel)
# y retornos que empeoran en esos episodios (solo para fijar el orden económico).
rng = np.random.default_rng(42)
n = 252 * 12
nivel = np.zeros(n)
for a in rng.choice(np.arange(252 * 3, n - 200), size=10, replace=False):
    nivel[a:a + rng.integers(60, 180)] += 2.5
ruido = np.zeros(n)
for t in range(1, n):
    ruido[t] = 0.97 * ruido[t - 1] + rng.normal(scale=0.25)
fechas = pd.bdate_range('2000-01-03', periods=n)
ret_demo = pd.Series(-0.001 * nivel + rng.normal(scale=0.01, size=n) * (1 + nivel), index=fechas)
X_demo = pd.DataFrame({'VIX_level_z': nivel + ruido}, index=fechas)

def fabrica_demo():
    return RuleVixThreshold(feature='VIX_level_z')

STEP_DEMO = 21
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    paneles_demo = {
        ctx: ev.walk_forward(fabrica_demo, X_demo, market_returns=ret_demo,
                             train_size=252 * 3, min_train=252 * 3, step=STEP_DEMO, context=ctx)
        for ctx in (0, 'auto')
    }
filas = []
for ctx, p in paneles_demo.items():
    salidas = p['state'].diff().eq(-1)
    en_borde = salidas & p['fold'].ne(p['fold'].shift())
    filas.append({'context': repr(ctx), 'dias_crisis': int(p['state'].sum()),
                  'salidas_de_crisis': int(salidas.sum()),
                  'salidas_en_borde_de_bloque': int(en_borde.sum())})
demo_contexto = pd.DataFrame(filas).set_index('context')
display(demo_contexto)
distintos = int((paneles_demo[0]['state'] != paneles_demo['auto']['state']).sum())
print(f"Días OOS con etiqueta distinta entre ambos protocolos: {distintos} de "
      f"{len(paneles_demo[0])} ({distintos / len(paneles_demo[0]):.1%}).")
print(f"Salidas de crisis en borde de bloque: {demo_contexto.loc['0', 'salidas_en_borde_de_bloque']} "
      f"en frío frente a {demo_contexto.loc[repr('auto'), 'salidas_en_borde_de_bloque']} con contexto.")

,dias_crisis,salidas_de_crisis,salidas_en_borde_de_bloque
context,,,
0,557,18,15
'auto',690,11,0


Días OOS con etiqueta distinta entre ambos protocolos: 149 de 2268 (6.6%).
Salidas de crisis en borde de bloque: 15 en frío frente a 0 con contexto.


**Lectura.** Con arranque en frío, buena parte de las salidas de crisis caen justo en
el primer día de un bloque: son artefactos del protocolo, no cambios de régimen. Con
contexto desaparecen y los episodios de crisis quedan enteros. En los datos reales,
ADR-003 documenta el mismo efecto sobre D01 en la pista A. Es una corrección de
causalidad **operativa** (el detector en producción nunca se reiniciaría), no un ajuste
para mejorar métricas: a cambio, la tasa de falsas alarmas puede subir ligeramente.

## 2. Qué se va a probar

Se ejecutan los doce detectores oficiales D1–D12, no la extensión HSMM D13. Cada
modelo conserva su familia y su lógica de Capa 1, pero recibe una configuración v2
declarada antes de observar resultados:

- los modelos multivariantes usan un núcleo fijo por pista, evitando entregar todas las
  columnas del panel (35 en A, 77 en B) a modelos de covarianza completa;
- los modelos de varianza y cambio usan el retorno logarítmico natural `SP500_ret`, derivado causalmente de la espina cruda;
- D1 usa VIX en B y volatilidad realizada en A, porque VIX no pertenece a la
  ventana histórica de A; esta variante queda marcada en todos los resultados;
- D2 usa equivalentes v2 de volatilidad, crédito, pendiente y drawdown;
- los detectores caros reducen la frecuencia de reestimación, pero continúan
  produciendo una etiqueta diaria causal dentro de cada bloque.

La tabla siguiente es el contrato efectivo. Si se cambia una fila, debe justificarse
antes de ejecutar y quedar registrada en el manifiesto.

In [4]:
tabla_A = specs_table('A').assign(pista='A')
tabla_B = specs_table('B').assign(pista='B')
display(pd.concat([tabla_A, tabla_B], ignore_index=True)[
    ['pista', 'id', 'detector', 'familia', 'n_features', 'step',
     'ventana_train', 'lento', 'variante_v2']
])


,pista,id,detector,familia,n_features,step,ventana_train,lento,variante_v2
0,A,D01,Regla de volatilidad/VIX,Reglas,1,21,expanding,False,A: umbral de volatilidad realizada; VIX no existe en la pista histórica
1,A,D02,Regla compuesta risk-off,Reglas,4,21,expanding,False,
2,A,D03,GMM (K=3),Clustering,9,21,expanding,False,
3,A,D04,HMM gaussiano (K=2),HMM,9,21,expanding,False,
4,A,D05,Markov-Switching varianza,Markov-Switching,1,63,expanding,True,
5,A,D06,GJR-GARCH-t,GARCH,1,21,expanding,False,
6,A,D07,CUSUM online robusto,Change-point,1,21,expanding,False,
7,A,D08,HMM t-Student (K=4),HMM,9,126,expanding,True,
8,A,D09,Statistical Jump Model,Jump model,9,21,expanding,False,
9,A,D10,Turbulencia Mahalanobis,Reglas multivariantes,4,21,expanding,False,


## 3. Cómo se implementa la comparación

Para cada pista y detector se aplica el mismo esquema:

1. se unen los bloques diario y mensual ya alineados por `03_preprocesado`;
2. se seleccionan únicamente las columnas declaradas para ese detector;
3. se fija una única rejilla completa usando la unión de features de los doce, para que todos compartan exactamente las mismas fechas OOS, sin imputar valores;
4. cada fold ajusta una instancia nueva usando exclusivamente observaciones
   anteriores al bloque OOS;
5. `predict_online` produce etiquetas diarias sin suavizado futuro, arrancando desde la
   cola del train del fold (contexto de ADR-003, §1.1);
6. las crisis y trampas se cargan desde `benchmark_spec.yaml` para la pista activa;
7. se guarda inmediatamente el resultado antes de continuar con el siguiente modelo.

La ventana inicial es común dentro de cada pista: ocho años en A y tres en B. La
ventana corta de B implica que la GFC queda en entrenamiento, no OOS; se declarará
como `NaN` y no se premiará ni penalizará. Los valores son configuración de benchmark,
no hiperparámetros elegidos mirando las métricas.

Consecuencias que conviene tener presentes al leer los notebooks de familia (05–11) y
`12_comparativa`:

- **Crisis realmente evaluadas.** El inicio del OOS de cada pista (tabla de §1) deja
  fuera del OOS las crisis que caen en el train inicial y deja otras solo parcialmente
  dentro. La celda siguiente cuenta, con el calendario del catálogo, cuántas crisis de
  cada pista se evalúan, cuáles quedan fuera o parciales y cuántos días hábiles aporta
  cada una al OOS (en A, `bear_1969_70` solo aporta sus últimos días; en B, la GFC
  queda en el train).
- **Frecuencia de refit.** D5 reestima cada 63 sesiones y D8/D11 cada 126 (D11 además con
  ventana móvil). No cambia las fechas OOS, pero sí `label_stability`, que compara la
  etiqueta de un bloque con la que le da el modelo siguiente: con refits más espaciados
  el modelo siguiente ha visto más datos nuevos.
- **Estado de los autómatas en cada refit.** *Texto anterior a ADR-003, conservado como
  histórico:* "Las reglas con histéresis (D1, D2, D6, D10) reinician su autómata en calma
  al comenzar cada bloque OOS; D7 sí propaga su CUSUM desde el train. Es causal, pero
  penaliza a esas reglas (ver limitaciones en 05)." **Vigente:** desde ADR-003 el
  protocolo propaga el estado con la cola del train (`context='auto'`, §1.1), así que
  ninguna regla se reinicia en calma en el borde de un bloque.

In [5]:
# Crisis del catálogo frente al inicio del OOS de cada pista (días hábiles aproximados,
# mismo cálculo que 12_comparativa §2.1; no lee results/).
import yaml
from regimenes.rutas import BENCHMARK_SPEC

SPEC = yaml.safe_load(BENCHMARK_SPEC.read_text(encoding='utf-8'))
filas_v = []
for track in ('A', 'B'):
    part = calendario[calendario.pista == track]
    if 'oos_inicio' not in part or part.oos_inicio.isna().all():
        continue
    oos_start, oos_end = pd.Timestamp(part.oos_inicio.iloc[0]), pd.Timestamp(part.oos_fin.iloc[0])
    for name, (a, b) in SPEC['crisis_windows'][f'pista_{track}'].items():
        a, b = pd.Timestamp(a), pd.Timestamp(b)
        dias = len(pd.bdate_range(a, b))
        dias_oos = len(pd.bdate_range(max(a, oos_start), min(b, oos_end))) if b >= oos_start else 0
        estado = 'fuera' if dias_oos == 0 else ('completa' if a >= oos_start else 'parcial')
        filas_v.append({'pista': track, 'crisis': name, 'inicio': a.date(), 'suelo': b.date(),
                        'dias_aprox': dias, 'dias_oos_aprox': dias_oos, 'estado': estado})
ventanas = pd.DataFrame(filas_v)
if len(ventanas):
    display(ventanas[ventanas.estado != 'completa'])
    for track, part in ventanas.groupby('pista'):
        evaluadas = part[part.estado != 'fuera']
        print(f"Pista {track}: {len(evaluadas)} de {len(part)} crisis evaluadas OOS "
              f"({(part.estado == 'parcial').sum()} parcial); fuera del OOS: "
              f"{', '.join(part.loc[part.estado == 'fuera', 'crisis']) or 'ninguna'}; parciales: "
              + (', '.join(f'{r.crisis} ({r.dias_oos_aprox} de {r.dias_aprox} días hábiles)'
                           for r in part[part.estado == 'parcial'].itertuples()) or 'ninguna'))
else:
    print('Sin paneles procesados: el calendario OOS no está disponible (ver gate de §6).')

,pista,crisis,inicio,suelo,dias_aprox,dias_oos_aprox,estado
0,A,credit_crunch_1966,1966-02-09,1966-10-07,173,0,fuera
1,A,bear_1969_70,1968-11-29,1970-05-26,388,11,parcial
18,B,gfc_2007_09,2007-10-09,2009-03-09,370,0,fuera


Pista A: 17 de 18 crisis evaluadas OOS (1 parcial); fuera del OOS: credit_crunch_1966; parciales: bear_1969_70 (11 de 388 días hábiles)
Pista B: 9 de 10 crisis evaluadas OOS (0 parcial); fuera del OOS: gfc_2007_09; parciales: ninguna


## 4. Caché con huella de contenido

Cada combinación pista×detector escribe dos artefactos en `results/benchmark/`: el CSV
de métricas (`metrics/<P>_<D>.csv`) y el panel OOS (`panels/<P>_<D>.parquet`, no
versionado). El CSV lleva una columna `cache_fingerprint`: el SHA-256 de un JSON con
todo lo que puede cambiar el resultado.

| componente | qué recoge |
|---|---|
| `cache_schema_version` | versión del formato de la huella (sube si cambia su definición) |
| `track`, `train_days`, `detector` | pista, train inicial y la `DetectorSpec` completa (features, parámetros, semilla, `step`, ventana) |
| `code_sha256` | hash de `walk_forward.py`, `metricas.py`, el registro, la clase base y los módulos oficiales de cada familia (`.py` normalizado a LF) |
| `benchmark_model_sha256` | hash del AST de las diez funciones del benchmark que determinan el resultado (`run_one`, `load_track_panel`, detección por evento, …) |
| `input_sha256` | contenido (no bytes) de `benchmark_spec.yaml`, los dos paneles procesados de la pista y el S&P 500 crudo |
| `runtime_versions` | versiones de Python y de las librerías numéricas del entorno |

Una caché solo se acepta si existen **ambos** artefactos y la huella del CSV coincide
con la recalculada; si no, se reajusta (o, en modo `CACHE_ONLY`, se marca `sin_cache`).
`manifest.json` registra las huellas de la ejecución completa y permite comprobar la
procedencia de las métricas versionadas en otra máquina sin datos locales
(`metrics_provenance`, usado en `12_comparativa`).

Cualquier cambio en esos archivos (incluidas sus rutas dentro de `src/regimenes/`) cambia
`code_sha256` y deja obsoletas todas las cachés hasta la re-ejecución completa.
La celda siguiente enumera los componentes locales de la huella sin leer `results/`.

In [6]:
print('Versión del esquema de huella:', bm_cache.CACHE_SCHEMA_VERSION)
print('Clase base resuelta:', bm_cache._resolved_detector_base().relative_to(ROOT).as_posix())
print('\nArchivos de código que entran en code_sha256:')
for path in bm_cache._cache_code_paths():
    print('  -', path.relative_to(ROOT).as_posix())
print('\nMódulos de los que se leen las funciones del modelo (AST):', list(bm_cache._MODEL_SOURCE_MODULES))
print('\nEntorno registrado en la huella:')
display(pd.Series(bm_cache._runtime_versions(), name='version').to_frame())

Versión del esquema de huella: 3
Clase base resuelta: src/regimenes/detectores/base.py

Archivos de código que entran en code_sha256:
  - src/regimenes/evaluacion/walk_forward.py
  - src/regimenes/evaluacion/metricas.py
  - src/regimenes/detectores/__init__.py
  - src/regimenes/detectores/registry.py
  - src/regimenes/detectores/base.py
  - src/regimenes/detectores/f1_reglas/__init__.py
  - src/regimenes/detectores/f1_reglas/rule_composite_riskoff.py
  - src/regimenes/detectores/f1_reglas/rule_vix_threshold.py
  - src/regimenes/detectores/f1_reglas/turbulence_mahalanobis.py
  - src/regimenes/detectores/f2_clustering/__init__.py
  - src/regimenes/detectores/f2_clustering/clustering_gmm.py
  - src/regimenes/detectores/f2_clustering/jump_model.py
  - src/regimenes/detectores/f3_hmm/__init__.py
  - src/regimenes/detectores/f3_hmm/_hmm_t_utils.py
  - src/regimenes/detectores/f3_hmm/_hmm_utils.py
  - src/regimenes/detectores/f3_hmm/hmm_gaussian_2s.py
  - src/regimenes/detectores/f3_hmm/hmm_t

,version
python,3.13.14
numpy,2.3.3
pandas,2.3.2
scipy,1.16.2
scikit-learn,1.8.0
hmmlearn,0.3.3
arch,8.0.0
torch,2.13.0
jumpmodels,0.1.1


## 5. Interruptores `EJECUTAR` / `CACHE_ONLY`

Por defecto el notebook es de **solo lectura**: `EJECUTAR = False` implica
`CACHE_ONLY = True`, de modo que abrirlo o ejecutar todas sus celdas no lanza horas de
cálculo ni sobrescribe `results/benchmark/`; solo verifica qué cachés superan la huella
de §4 en esta máquina. Para recalcular hay que poner `EJECUTAR = True` de forma
explícita (o, mejor, usar la CLI de §7). Es la misma convención de los notebooks de
familia 05–11.

In [7]:
# Interruptores deliberados. Con CACHE_ONLY=True, abrir o ejecutar todas las celdas NO
# lanza horas de cálculo ni sobrescribe results/: solo se leen las cachés cuya huella
# (código + datos + entorno) se verifica en esta máquina. Para recalcular (~4 h en serie)
# hay que poner EJECUTAR=True (equivale al antiguo CACHE_ONLY=False) de forma explícita.
EJECUTAR = False              # True = ajusta lo que falte y escribe en results/benchmark
CACHE_ONLY = not EJECUTAR     # True = solo lectura; False = ajusta lo que falte
RUN_BENCHMARK = True          # False = salta por completo la celda de §7
FORCE = False                 # False reutiliza cada CSV ya terminado
N_JOBS = 1                    # >1 = un proceso por detector (ver §7); solo si EJECUTAR=True
TRACKS = ('A', 'B')
DETECTOR_IDS = tuple(f'D{i:02d}' for i in range(1, 13))
TRAIN_DAYS = dict(bm.DEFAULT_TRAIN_DAYS)   # {'A': 252 * 8, 'B': 252 * 3}
OUTPUT_DIR = RESULTS_BENCHMARK

print('EJECUTAR =', EJECUTAR, '| CACHE_ONLY =', CACHE_ONLY, '| RUN_BENCHMARK =', RUN_BENCHMARK,
      '| N_JOBS =', N_JOBS)
print('Detectores:', DETECTOR_IDS)
print('Train inicial:', TRAIN_DAYS)
print('Salida:', OUTPUT_DIR.relative_to(ROOT))

EJECUTAR = False | CACHE_ONLY = True | RUN_BENCHMARK = True | N_JOBS = 1
Detectores: ('D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12')
Train inicial: {'A': 2016, 'B': 756}
Salida: results\benchmark


## 6. Gate previo: datos, columnas y dependencias

El benchmark no debe empezar parcialmente sin avisar. Este gate comprueba que los
cuatro Parquet de `data/processed/` existen, que cada configuración encuentra sus
features y que están instaladas las dependencias opcionales. En particular, D6
requiere `arch` y D9 requiere `jumpmodels`.

Si faltan los paneles —situación normal después de clonar porque están ignorados por
Git— hay que ejecutar primero `python -m regimenes.datos` y después `03_preprocesado.ipynb`.

In [8]:
preflight = pd.concat([bm.preflight(track) for track in TRACKS], ignore_index=True)
display(preflight)
HAY_DATOS = bool(preflight['datos_ok'].all())
GATE_OK = bool(HAY_DATOS and preflight['features_ok'].all() and preflight['dependencia_ok'].all())
print('Gate completo:', 'OK' if GATE_OK else 'PENDIENTE')
if not HAY_DATOS:
    print('Acción: ejecutar python -m regimenes.datos y después notebooks/03_preprocesado.ipynb.')
if HAY_DATOS and not preflight['dependencia_ok'].all():
    print('Acción: pip install -r requirements.txt')

,pista,id,datos_ok,features_ok,dependencia_ok,detalle
0,A,D01,True,True,True,OK
1,A,D02,True,True,True,OK
2,A,D03,True,True,True,OK
3,A,D04,True,True,True,OK
4,A,D05,True,True,True,OK
5,A,D06,True,True,True,OK
6,A,D07,True,True,True,OK
7,A,D08,True,True,True,OK
8,A,D09,True,True,True,OK
9,A,D10,True,True,True,OK


Gate completo: OK


## 7. Ejecución reanudable y CLI paralela

La celda siguiente recorre la matriz pista × detector. `run_status.csv` se actualiza
tras cada intento y cada CSV individual actúa como *checkpoint*. Si una ejecución se
interrumpe, se vuelve a lanzar con `FORCE=False` y continúa desde el primer resultado
ausente. Los errores se registran y no borran resultados previos. Con `CACHE_ONLY=True` la celda
es de solo lectura: no ajusta modelos ni reescribe `manifest.json` ni `run_status.csv`. Si alguna
combinación queda `sin_cache`, la celda lo dice e imprime el comando exacto para regenerarla.

Para una prueba corta puede cambiarse temporalmente, por ejemplo,
`DETECTOR_IDS = ('D01', 'D07')`. La ejecución oficial debe restaurar los doce.


### Re-ejecución completa en paralelo (recomendada tras ADR-003)

La huella de caché cambió en ADR-003 (esquema 3: contenido de los parquet en vez de
bytes, `detector_base` realmente importado, columnas `det_*` de detección por evento),
así que **todas** las cachés versionadas quedan invalidadas y hay que recalcular una vez.
Lo mismo ocurre con cualquier cambio de código, datos o entorno que entre en la huella (§4).
Lo más cómodo es hacerlo fuera del notebook, desde la raíz del repo:

```bash
# matriz completa, 4 procesos (lentos primero); manifest.json y run_status.csv los
# escribe solo el proceso principal, cada hijo escribe su CSV, su panel y status/<P>_<D>.json
python -m regimenes.benchmark --track A B --jobs 4

# alternativa: un detector por terminal (no toca manifest/run_status) y consolidar al final
python -m regimenes.benchmark --track A --detector D04
python -m regimenes.benchmark --track A --detector D09
python -m regimenes.benchmark --consolidate --track A B
```

`--jobs` limita cada hijo a 1 hilo BLAS/OpenMP (`--threads-per-job` para cambiarlo) y
`--force` ignora la caché. Después basta con volver a abrir este notebook con
`EJECUTAR=False` (`CACHE_ONLY=True`; leerá las cachés nuevas) y ejecutar los notebooks de
familia 05–11 y `12_comparativa`. Desde el notebook equivale a `EJECUTAR=True` y `N_JOBS=4`.

| opción de `python -m regimenes.benchmark` | efecto |
|---|---|
| `--track A B` | pistas a ejecutar |
| `--detector D04 D09` | solo esas combinaciones; con `--jobs 1` no escribe `manifest`/`run_status` (consolidar al final) |
| `--jobs N` | `N` procesos (`spawn`), lentos primero; el proceso principal es el único que escribe `manifest.json` y `run_status.csv` |
| `--threads-per-job K` | hilos BLAS/OpenMP por hijo (defecto 1; 0 = no tocar el entorno) |
| `--cache-only` | solo verifica la caché (equivale a `EJECUTAR=False`) |
| `--force` | ignora la caché y reajusta |
| `--consolidate` | reconstruye `manifest.json` y `run_status.csv` desde `status/*.json` |
| `--output DIR` | directorio de salida (defecto `results/benchmark`) |

In [9]:
if RUN_BENCHMARK:
    if not GATE_OK:
        raise RuntimeError('El gate previo no está en verde; corrige datos/dependencias antes de ejecutar.')
    metrics_run, run_status = bm.run_benchmark(
        tracks=TRACKS,
        detector_ids=DETECTOR_IDS,
        output_dir=OUTPUT_DIR,
        train_days=TRAIN_DAYS,
        force=FORCE,
        continue_on_error=True,
        cache_only=CACHE_ONLY,
        n_jobs=N_JOBS,
    )
    if CACHE_ONLY and (run_status['estado'] != 'cache').any():
        print('Modo solo-caché: hay combinaciones sin caché verificable en esta máquina '
              '(huella distinta o paneles OOS ausentes). No se ha recalculado nada; '
              '12_comparativa usará las métricas versionadas y lo indicará.')
        faltan = run_status[run_status['estado'] != 'cache']
        print('Para regenerarlas: python -m regimenes.benchmark --track '
              + ' '.join(sorted(faltan['pista'].unique())) + ' --detector '
              + ' '.join(sorted(faltan['id'].unique())) + ' --jobs 4')
    display(run_status)
else:
    print('Ejecución desactivada. Cambia RUN_BENCHMARK=True cuando el gate esté en verde.')

,pista,id,estado,segundos,detalle
0,A,D01,cache,0.0,caché verificada: results\benchmark\metrics\A_D01.csv
1,A,D02,cache,0.0,caché verificada: results\benchmark\metrics\A_D02.csv
2,A,D03,cache,0.0,caché verificada: results\benchmark\metrics\A_D03.csv
3,A,D04,cache,0.0,caché verificada: results\benchmark\metrics\A_D04.csv
4,A,D05,cache,0.0,caché verificada: results\benchmark\metrics\A_D05.csv
5,A,D06,cache,0.0,caché verificada: results\benchmark\metrics\A_D06.csv
6,A,D07,cache,0.0,caché verificada: results\benchmark\metrics\A_D07.csv
7,A,D08,cache,0.0,caché verificada: results\benchmark\metrics\A_D08.csv
8,A,D09,cache,0.0,caché verificada: results\benchmark\metrics\A_D09.csv
9,A,D10,cache,0.0,caché verificada: results\benchmark\metrics\A_D10.csv


## 8. Inventario de resultados disponibles

Esta sección puede ejecutarse aunque el benchmark esté incompleto. Permite ver qué
detectores terminaron, cuáles se recuperaron de caché validada por huella y qué errores necesitan una
repetición. No construye rankings parciales silenciosamente: la comparación del
notebook 12 mostrará expresamente cuántas filas hay por pista.

In [10]:
status_path = OUTPUT_DIR / 'run_status.csv'
if status_path.exists():
    status = pd.read_csv(status_path)
    display(status)
    print('\nConteo por estado:')
    display(status.groupby(['pista', 'estado']).size().rename('n').reset_index())
else:
    print('Todavía no existe run_status.csv.')

metric_files = sorted((OUTPUT_DIR / 'metrics').glob('[AB]_D*.csv'))
print(f'CSV de métricas disponibles: {len(metric_files)}/24')
for path in metric_files:
    print(' -', path.relative_to(ROOT))


,pista,id,estado,segundos,detalle,inicio_utc,fin_utc,pid
0,A,D01,ok,41.714511,results\benchmark\metrics\A_D01.csv,2026-09-29T12:41:39.203682+00:00,2026-09-29T12:42:21.521247+00:00,19048
1,A,D02,ok,26.595666,results\benchmark\metrics\A_D02.csv,2026-09-29T12:42:21.524313+00:00,2026-09-29T12:42:48.459208+00:00,19048
2,A,D03,ok,578.525791,results\benchmark\metrics\A_D03.csv,2026-09-29T12:42:34.735508+00:00,2026-09-29T12:52:13.695644+00:00,12484
3,A,D04,ok,6562.037653,results\benchmark\metrics\A_D04.csv,2026-09-29T12:42:48.463384+00:00,2026-09-29T14:32:10.777048+00:00,19048
4,A,D05,ok,4064.725523,results\benchmark\metrics\A_D05.csv,2026-09-29T12:41:39.136973+00:00,2026-09-29T13:49:24.380091+00:00,23832
5,A,D06,ok,190.783888,results\benchmark\metrics\A_D06.csv,2026-09-29T12:44:36.333606+00:00,2026-09-29T12:47:47.601280+00:00,9260
6,A,D07,ok,105.482116,results\benchmark\metrics\A_D07.csv,2026-09-29T12:45:33.271306+00:00,2026-09-29T12:47:19.294268+00:00,10956
7,A,D08,ok,1776.028744,results\benchmark\metrics\A_D08.csv,2026-09-29T12:41:39.110854+00:00,2026-09-29T13:11:15.636721+00:00,19216
8,A,D09,ok,5676.076887,results\benchmark\metrics\A_D09.csv,2026-09-29T12:47:19.297285+00:00,2026-09-29T14:21:55.740093+00:00,10956
9,A,D10,ok,2612.303194,results\benchmark\metrics\A_D10.csv,2026-09-29T12:47:31.432448+00:00,2026-09-29T13:31:04.061577+00:00,23996



Conteo por estado:


,pista,estado,n
0,A,ok,12
1,B,ok,12


CSV de métricas disponibles: 24/24
 - results\benchmark\metrics\A_D01.csv
 - results\benchmark\metrics\A_D02.csv
 - results\benchmark\metrics\A_D03.csv
 - results\benchmark\metrics\A_D04.csv
 - results\benchmark\metrics\A_D05.csv
 - results\benchmark\metrics\A_D06.csv
 - results\benchmark\metrics\A_D07.csv
 - results\benchmark\metrics\A_D08.csv
 - results\benchmark\metrics\A_D09.csv
 - results\benchmark\metrics\A_D10.csv
 - results\benchmark\metrics\A_D11.csv
 - results\benchmark\metrics\A_D12.csv
 - results\benchmark\metrics\B_D01.csv
 - results\benchmark\metrics\B_D02.csv
 - results\benchmark\metrics\B_D03.csv
 - results\benchmark\metrics\B_D04.csv
 - results\benchmark\metrics\B_D05.csv
 - results\benchmark\metrics\B_D06.csv
 - results\benchmark\metrics\B_D07.csv
 - results\benchmark\metrics\B_D08.csv
 - results\benchmark\metrics\B_D09.csv
 - results\benchmark\metrics\B_D10.csv
 - results\benchmark\metrics\B_D11.csv
 - results\benchmark\metrics\B_D12.csv


## 9. Cierre

Este notebook termina cuando existen 24 filas válidas —12 detectores por dos pistas—
o cuando cualquier excepción queda documentada como incompatibilidad explícita. No
se interpreta aquí el ganador: cada notebook de familia (05–11) analiza sus detectores y
`12_comparativa.ipynb` carga los artefactos, verifica comparabilidad dentro de cada pista
y construye la evidencia visual.

La prueba pseudolive estricta es la validación final de la señal de régimen de la Fase 1 (la que
alimentará a los agentes del TFM), no la meta del TFM. Será posterior a esta comparación y utilizará
un tramo final congelado que no intervenga en la selección.